# 07 — Five-fold classifier and feature selection benchmark

Classification only. All fold-specific feature transformations, PCA, L1 selection and Ridge ranking are fit on past training rows. No regression models.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.classifier_benchmark import eligible_universe, run_benchmark, plot_results

# Jupyter may start in either the repository root or its notebooks/ directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the stock_ml_dynamic repository (pyproject.toml not found).")
INTERVAL = "1d"
REQUIRED = ("master_labeled.parquet", "latest_inference.parquet", "transformers.joblib", "manifest.json")
CANDIDATES = [
    ROOT / "artifacts" / "master_tables" / INTERVAL,
    ROOT / "notebooks" / "artifacts" / "master_tables" / INTERVAL,
]
MASTER_DIR = next((p for p in CANDIDATES if all((p / f).is_file() for f in REQUIRED)), None)
if MASTER_DIR is None:
    searched = "\n".join(f"  - {p}" for p in CANDIDATES)
    raise FileNotFoundError(
        "Master-table artifacts not found. Searched:\n" + searched +
        "\nRun notebooks/03_universe_master_table.ipynb with SAVE_RESULTS=True "
        "to create the four required files, then rerun this cell."
    )
print("Loading saved master table from:", MASTER_DIR)
master, latest, _, manifest = load_master(MASTER_DIR)
features = pd.concat([master, latest], ignore_index=True)
eligible, universe = eligible_universe(features)
display(universe.loc[~universe.eligible])
print(f"Eligible tickers: {eligible.symbol.nunique()} / {features.symbol.nunique()}")


## Experiment settings

Start with `FAST_MODE=True` for a smoke test, then change to `False` for the full 5-fold × 4 feature schemes × 7 classifiers benchmark. Training is capped at 120,000 rows per fold to control compute. All test rows are evaluated. Expect the full benchmark to take substantial time.

In [ ]:
N_FOLDS = 5
FAST_MODE = True  # Set False for full comparison after confirming a successful run.
SCHEMES = ('all', 'pca_95', 'lasso_l1', 'ridge_l2')
MODELS = ('prior_baseline', 'logistic_l2', 'logistic_l1', 'random_forest',
          'gaussian_nb', 'hist_gradient_boosting', 'mlp_neural_network')
if FAST_MODE:
    SCHEMES = ('all', 'pca_95')
    MODELS = ('prior_baseline', 'logistic_l2', 'gaussian_nb', 'mlp_neural_network')
OUTPUT = ROOT / 'artifacts' / 'classifier_benchmark' / INTERVAL
print('Output:', OUTPUT, '| runs:', N_FOLDS * len(SCHEMES) * len(MODELS))

In [ ]:
scores, roc, universe = run_benchmark(features, n_folds=N_FOLDS, schemes=SCHEMES,
    models=MODELS, max_train_rows=120000, output_dir=OUTPUT)
display(scores.tail(12))

## Accuracy, AUC-ROC and training time

Multiclass AUC is macro one-vs-rest. The plotted ROC curves average class-specific TPR across classes and folds on a common FPR grid; the displayed AUC values are the mean fold-level sklearn OVR AUC.

In [ ]:
summary = plot_results(scores, roc, OUTPUT)
display(summary)
display(scores.pivot_table(index=['scheme','model'],values=['accuracy','balanced_accuracy','roc_auc_ovr_macro','fit_seconds'],aggfunc=['mean','std']).round(4))
print('Saved metrics, ROC coordinates and plots:', OUTPUT)

## Interpretation and safeguards

- A baseline near 33% is **not guaranteed** on future folds; compare actual fold baseline accuracy.
- L1 logistic is sparse; Ridge (L2) is **not** sparse, so we use its coefficient magnitudes to rank and retain the strongest half of features.
- PCA retains 95% training variance; it is not supervised feature selection.
- The MLP uses random early-stopping samples from the **training fold only**; outer test periods remain untouched.
- These five folds are exploratory model selection, **not** an untouched final test. Select a candidate, then evaluate it on a separately reserved future period.
- `run_benchmark` samples up to 120,000 training rows per fold for compute control, never the test set.
- ROC-AUC and accuracy are diagnostic, not evidence of profitable trading.